# Phi-4-mini-instruct

## 一、项目简介

### 1.1 官方项目简介

Phi-4-mini-instruct 是微软推出的紧凑型指令微调语言模型，基于 Phi-3 架构。支持 128K 上下文窗口，MIT 许可证开源。

| 属性 | 值 |
|------|-----|
| 参数量 | ~4B |
| 上下文长度 | 128K tokens |
| 架构 | Phi3ForCausalLM |
| 许可证 | MIT |
| 显存需求 | ~8GB（FP16） |

### 1.2 本项目简介

本项目在趋动云平台部署 Phi-4-mini-instruct 推理服务，提供 Gradio WebUI 对话界面。模型懒加载，首次对话约 60 秒加载完成后即可流畅对话。

| 属性 | 值 |
|------|-----|
| 推理框架 | PyTorch 2.5.1 + Transformers 5.9.0 |
| Web UI | Gradio 4.7.1 |
| Python | 3.10 |
| CUDA | 12.1 |
| 模型格式 | FP16 Safetensors（2 分片，~7.3GB） |

## 二、官方链接

- **HuggingFace**: https://huggingface.co/microsoft/Phi-4-mini-instruct
- **微软 Phi 系列**: https://azure.microsoft.com/en-us/products/phi

## 三、算力推荐

| 场景 | 推荐规格 | 显存 |
|------|------|:--:|
| 最低运行 | T4 | 8GB |
| 流畅体验 | A10 / A100 | 16GB+ |
| 长上下文 | A100 40GB | 40GB |

## 四、使用说明

必须先开放 **7860** 端口，一键启动如下：

```bash
cd /gemini/code && bash start.sh
```

### Web 使用界面效果展示

![SP](SP.png)

### Jupyter 内手动调用

如需在 Jupyter 中直接调用模型：

In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_DISABLE_SSL_VERIFY"] = "1"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "/gemini/pretrain/Phi-4-mini-instruct-model"

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    local_files_only=True,
)
model = model.cuda()
print(f"Model loaded on: {model.device}")

In [ ]:
def chat(user_message, history=None, temperature=0.7, max_new_tokens=2048):
    messages = []
    if history:
        for q, a in history:
            messages.append({"role": "user", "content": q})
            messages.append({"role": "assistant", "content": a})
    messages.append({"role": "user", "content": user_message})

    tokenized = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
    ).to(model.device)

    input_ids = tokenized["input_ids"]
    input_len = input_ids.shape[-1]

    with torch.no_grad():
        generated = model.generate(
            **tokenized,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )

    output_tokens = generated[0][input_len:]
    response = tokenizer.decode(output_tokens, skip_special_tokens=True)
    return response

result = chat("你好！请用中文做一个简短的自我介绍。")
print(result)